# 03 · USDA NASS cattle inventory and cow slaughter

**Milestone 1, items 1.4 and 1.5** · see `docs/MILESTONES.md`

## What we're looking for

This notebook gets the data to test the cause chain in decision D17:

**drought → less grazing → ranchers sell off cows → smaller herd → fewer cattle →
higher beef prices**

Two kinds of data from USDA's National Agricultural Statistics Service (NASS):

| Data | What it shows | Link in the chain |
|---|---|---|
| **Cattle inventory** | How many cattle and beef cows are on U.S. farms and ranches, counted each January 1 (and mid-year) | "smaller herd" |
| **Cow slaughter** | How many cows are slaughtered each month | "ranchers sell off cows": a sell-off shows up here first, as a spike |

**Why cows specifically?** Steers and heifers raised for beef are slaughtered every year
as normal business. **Cows** are the breeding herd: every cow slaughtered is one fewer
calf next year and the year after. A jump in cow slaughter means ranchers are shrinking
their herds, which is exactly what happens in a drought.

**Why start in 2010, not 2015?** The cattle cycle is slow, so we need the years before
2015 for context. Ranchers sold off cows in the 2011–2013 drought (mainly Texas and the
southern Plains), rebuilt the herd from about 2014 to 2019, then sold off again. Seeing
the earlier cycle helps us judge whether the recent one is unusual (decision D24).

**Like notebook 02, this has two parts.** Part 1 downloads and takes an inventory: NASS
has many similar-sounding series, so we'll choose which ones to use together before
Part 2 builds the tables.

**Steps (Part 1)**
1. Setup and API key
2. The two queries
3. Check the record counts (NASS limits each request to 50,000 rows)
4. Download and save the raw responses
5. Load into tables
6. Inventory: every series, its time period and its coverage

## Step 1 · Setup and API key

Same approach as notebook 01: the key is read from `.env` and never printed.

In [ ]:
import json
import os
from datetime import date
from pathlib import Path

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv()
load_dotenv(env_path)
NASS_API_KEY = os.getenv("NASS_API_KEY", "").strip()

PROJECT_ROOT = Path(env_path).parent if env_path else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / f"nass_{date.today():%Y%m%d}"
RAW_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)

print("Project root:", PROJECT_ROOT)
print("API key found:", bool(NASS_API_KEY), "| length:", len(NASS_API_KEY))

### How to read it
**API key found** should be `True`. If it's `False`, check the `.env` line reads exactly
`NASS_API_KEY=yourkey`, then restart the kernel.

## Step 2 · The two queries

NASS's Quick Stats API is searched with filters. Each filter narrows the results:

| Filter | Value | Meaning |
|---|---|---|
| `commodity_desc` | CATTLE | Cattle data only |
| `statisticcat_desc` | INVENTORY / SLAUGHTERED | Head counts on farms, or animals slaughtered |
| `agg_level_desc` | NATIONAL | U.S. totals, not state or county |
| `source_desc` | SURVEY | NASS's regular surveys (not the 5-yearly Census of Agriculture) |
| `year__GE` | 2010 | Years greater than or equal to 2010 |
| `freq_desc` | MONTHLY | Slaughter only: monthly figures |

We don't narrow further (for example to "beef cows" only) yet, because we want to see
every series NASS has before choosing.

In [ ]:
NASS_BASE = "https://quickstats.nass.usda.gov/api"

QUERIES = {
    "inventory": {
        "commodity_desc": "CATTLE",
        "statisticcat_desc": "INVENTORY",
        "agg_level_desc": "NATIONAL",
        "source_desc": "SURVEY",
        "year__GE": "2010",
    },
    "slaughter": {
        "commodity_desc": "CATTLE",
        "statisticcat_desc": "SLAUGHTERED",
        "agg_level_desc": "NATIONAL",
        "source_desc": "SURVEY",
        "freq_desc": "MONTHLY",
        "year__GE": "2010",
    },
}

pd.DataFrame(QUERIES).fillna("")

## Step 3 · Check the record counts

NASS refuses any request that would return more than 50,000 rows, so we ask how many
rows each query would return before downloading. `get_counts` is a separate NASS
endpoint that returns only the count.

In [ ]:
def nass_get(endpoint, params):
    """Call a NASS Quick Stats endpoint and return the parsed JSON."""
    full = {"key": NASS_API_KEY, "format": "JSON", **params}
    resp = requests.get(f"{NASS_BASE}/{endpoint}/", params=full, timeout=120)
    if resp.status_code != 200:
        # NASS explains errors in the body; don't print resp.url, it contains the key
        raise RuntimeError(f"NASS {endpoint} failed ({resp.status_code}): {resp.text[:300]}")
    return resp.json()

for name, params in QUERIES.items():
    count = int(nass_get("get_counts", params)["count"])
    flag = "OK" if count <= 50_000 else "TOO MANY - narrow the query"
    print(f"{name:10} {count:>8,} rows  {flag}")

### How to read it
One line per query with the number of rows it would return. Both should say **OK**.
If one says TOO MANY, tell me and we'll split it (for example by year).

## Step 4 · Download and save

Same rule as before: save exactly what NASS sends, untouched, in a dated folder.

In [ ]:
raw = {}
for name, params in QUERIES.items():
    data = nass_get("api_GET", params)
    raw[name] = data["data"]
    path = RAW_DIR / f"{name}.json"
    path.write_text(json.dumps(data, indent=1))
    print(f"{name:10} {len(raw[name]):>8,} rows  ->  {path.relative_to(PROJECT_ROOT)}")

## Step 5 · Load into tables

Each NASS row has about 40 fields. We keep the ones that identify the series and its
time period:

| Field | Meaning |
|---|---|
| `short_desc` | The series name, e.g. *CATTLE, COWS, BEEF - INVENTORY* |
| `year` | Year |
| `reference_period_desc` | The period within the year: *FIRST OF JAN*, *FIRST OF JUL*, or a month (*JAN*, *FEB*…) |
| `freq_desc` | ANNUAL, POINT IN TIME, MONTHLY… |
| `Value` | The number, as text |
| `unit_desc` | Units, e.g. HEAD |

**About `Value`:** NASS writes numbers with thousands commas (`28,700,000`, the same
problem as ERS) and uses codes for values it can't publish: `(D)` = withheld to protect
individual operations, `(NA)` = not available, `(Z)` = less than half the unit shown. We
strip the commas and turn the codes into missing values, keeping the original text in
`value_raw` so nothing is lost.

In [ ]:
KEEP = ["short_desc", "year", "reference_period_desc", "freq_desc",
        "begin_code", "end_code", "Value", "unit_desc", "class_desc", "domain_desc"]

def to_frame(rows, query_name):
    df = pd.DataFrame(rows)
    df = df[[col for col in KEEP if col in df.columns]].copy()
    df["value_raw"] = df["Value"].astype(str).str.strip()
    df["value"] = pd.to_numeric(df["value_raw"].str.replace(",", "", regex=False), errors="coerce")
    df["year"] = df["year"].astype(int)
    df["query"] = query_name
    return df.drop(columns="Value")

tables = {name: to_frame(rows, name) for name, rows in raw.items()}

for name, df in tables.items():
    codes = df.loc[df["value"].isna(), "value_raw"].value_counts().to_dict()
    print(f"{name:10} {len(df):>7,} rows | {df['short_desc'].nunique():>3} series | "
          f"{df['year'].min()}-{df['year'].max()} | not numeric: {codes or 'none'}")

### How to read it
For each query: rows, number of distinct series, the year range, and any values that
weren't numbers (with their codes and how many times each appears).

## Step 6 · Inventory of series

**What we're looking for:** the exact names of the series we need:
- **Inventory:** total cattle and **beef cows** on January 1 (and July 1, if published),
  every year from 2010.
- **Slaughter:** monthly **cow** slaughter, ideally split into beef cows and dairy cows,
  since only beef cows matter for the beef herd.

One series can have several time periods (January 1 and July 1, or annual totals and
months), so we list each series with each period it comes in.

In [ ]:
def inventory_of(df):
    return (
        df.groupby(["short_desc", "freq_desc", "unit_desc"], dropna=False)
        .agg(periods=("reference_period_desc", lambda s: ", ".join(sorted(s.unique())[:6])
                      + (" ..." if s.nunique() > 6 else "")),
             first_year=("year", "min"), last_year=("year", "max"),
             rows=("value", "size"), numeric=("value", "count"))
        .reset_index()
    )

inventory_of(tables["inventory"])

### How to read it
Every **inventory** series: its name, frequency, units, which periods it covers (for
example *FIRST OF JAN, FIRST OF JUL*), the first and last year, how many rows, and how
many of those have a usable number.

In [ ]:
inventory_of(tables["slaughter"])

### How to read it
The same for **slaughter**. Look for series whose names start *CATTLE, COWS* (all cows)
and, if present, separate *beef* and *dairy* cow series. Names may also mention
*COMMERCIAL* (all plants) or *FEDERALLY INSPECTED* (federally inspected plants only).

### What it means
_To fill in together after this run: which series we keep for the herd and for cow
slaughter, and why._

## What's next and why

**Part 2 of this notebook:** with the series chosen, build two tables (yearly herd size,
monthly cow slaughter), check coverage, and chart them next to the beef price from
notebook 01. That's the first direct look at whether the herd shrank and cow slaughter
spiked before beef prices climbed.